<h1 class="alert alert-info"><center>Activité : algorithme des k plus proches voisins</center></h1>

Le but de cette activité est de programmer, sur un exemple simple, l'algorithme des k plus proches voisins.

On dispose d'un fichier contenant 15000 noms de communes de la France métropolitaine (hors Corse) avec leur coordonnées GPS et leur département.

Le but de ce TP est de construire à partir de ce fichier une liste de données d'apprentissage qui, une fois acquise, permettra de prédire, à l'aide de l'algorithme des k plus proches voisins, le département de n'importe quelle des 34466 communes de la France métropolitaine (hors Corse) à partir de ses coordonnées GPS.

<h2 class="alert alert-info">Exercice 1 : récupération et analyse des données</h2>

Les données qui vont nous permettre de construire notre liste d'apprentissage sont fournis par le fichier `Liste_communes_coordonnees_GPS.csv`.

1. Exécuter la cellule suivante pour construire la liste d'apprentissage.

In [ ]:
import csv

with open('Liste_communes_coordonnees_GPS.csv', 'r') as fichier:
    liste_apprentissage = list(csv.DictReader(fichier, delimiter=';'))

2. Exécuter la cellule suivante. Que représente la valeur renvoyée ? Est-ce correct ?

In [ ]:
len(liste_apprentissage)

<p style="font-style:italic; color:gray;">Ecrire la réponse ici</p>

3. Exécuter la cellule suivante. Que représente la valeur renvoyée et quel est son type ? En déduire les descripteurs du fichier csv.

In [ ]:
liste_apprentissage[0]

<p style="font-style:italic; color:gray;">Ecrire la réponse ici</p>

4. Compléter l'instruction suivante pour obtenir le département du centième enregistrement du fichier csv.

In [ ]:
liste_apprentissage[99][...]

5. Compléter l'instruction suivante pour obtenir les coordonnées GPS du millième enregistrement du fichier csv.

In [ ]:
liste_apprentissage[...][...]

Quel est le type de la donnée renvoyée ? Quel est son formatage ? Est-ce pertinent ?

<p style="font-style:italic; color:gray;">Ecrire la réponse ici</p>

<h2 class="alert alert-info">Exercice 2 : gestion des coordonnées GPS</h2>

1. Comme nous l'avons vu à la dernière question de l'exercice 1, les coordonnées GPS ne sont pas formatées correctement pour pouvoir être facilement utilisable.

La fonction `coordonnees_gps_formatees` ci-dessous prend en paramètre une chaîne de caractères de la forme `'xxxxxx, xxxxxx'` correspondant à des coordonnées GPS. Elle renvoie ces coordonnées GPS sous la forme d'un tuple de deux nombres flottants.

Par exemple : `coordonnees_gps_formatees('50.731528018, 2.58086143092')` renvoie le tuple `(50.731528018, 2.58086143092)`.

Compléter cette fonction.

In [ ]:
# on pourra exécuter l'instruction suivante pour obtenir la documentation de la fonction split
help(str.split)

In [ ]:
def coordonnees_gps_formatees(coordonnees):
    latitude, longitude = coordonnees.split(...)
    return float(...), ...

Les trois cellules ci-dessous permettent de tester votre code. Elles doivent toutes renvoyer `True` lors de leur exécution.

In [ ]:
coordonnees_gps_formatees('50.731528018, 2.58086143092') == (50.731528018, 2.58086143092)

In [ ]:
coordonnees_gps_formatees(liste_apprentissage[0]['Coordonnees_gps']) == (43.762029942, 0.297842185088)

In [ ]:
coordonnees_gps_formatees(liste_apprentissage[99]['Coordonnees_gps']) == (49.7754431249, 0.633291896241)

2. Maintenant que les coordonnées GPS sont formatées correctement sous la forme d'un tuple `(latitude, longitude)`, les deux valeurs étant données sous forme de nombres flottants, il s'agit à présent de savoir calculer la distance entre deux points donnés par ce type de coordonnées.

Pour ce faire, on utilise une <a href='https://geodesie.ign.fr/contenu/fichiers/Distance_longitude_latitude.pdf'>formule fondamentale</a> en trigonométrie sphérique :
 - soient $A$ et $B$ deux points donnés respectivement par leur latitude $\varphi_A$ et $\varphi_B$ exprimées en radians et leur longitude $\lambda_A$ et $\lambda_B$ exprimées également en radians ;
 - la longueur en radians de l'arc passant par $A$ et $B$ est : $\arccos (\sin(\varphi_A)\sin(\varphi_B) + \cos(\varphi_A)\cos(\varphi_B)\cos(\lambda_A-\lambda_B))$ ;
 - la longueur en km entre les deux points $A$ et $B$ est alors obtenue en multipliant la valeur précédente par le rayon conventionnel de la Terre en kilomètres, soit $6371$.
 
La fonction `distance` ci-dessous prend en paramètre deux tuples `gps_1` et `gps_2` de deux nombres flottants correspondant à des coordonnées GPS de deux points. Elle renvoie la distance, **arrondie au kilomètre**, entre ces deux points.

Compléter cette fonction.

In [ ]:
from math import pi, sqrt, cos, sin, acos

def distance(gps_1, gps_2):
    # conversion des latitudes et longitudes en radians
    lat_rad_1, long_rad_1 = gps_1[0]*pi/180, gps_1[1]*pi/180
    lat_rad_2, long_rad_2 = gps_2[0]*pi/180, gps_2[1]*pi/180
    
    # calcul de la longueur de l'arc en radians
    longueur_arc_rad = acos(sin(lat_rad_1)*sin(lat_rad_2) + cos(lat_rad_1)*cos(lat_rad_2)*cos(long_rad_1-long_rad_2))
    
    # renvoie la longueur de l'arc en km, arrondie à l'unité
    return int(...)

Les deux cellules ci-dessous permettent de tester votre code. Elles doivent toutes renvoyer `True` lors de leur exécution.

In [ ]:
distance((50.731528018, 2.58086143092), (43.762029942, 0.297842185088)) == 793

In [ ]:
distance((50.731528018, 2.58086143092), (49.7754431249, 0.633291896241)) == 174

<h2 class="alert alert-info">Exercice 3 : construction d'une liste triée</h2>

Lorsque l'on applique l'algorithme des k plus proches voisins, on calcule toutes les distances entre notre élément à classer et ceux que l'on possède dans la liste d'apprentissage. On stocke alors dans une liste des tuples contenant, pour chacun de nos éléments de la liste d'apprentissage, la distance avec notre élément à classer et les informations nécessaires à la décision finale (dans ce TP, il s'agit du département), puis on trie notre liste par ordre croissant sur les distances afin de sélectionner plus facilement nos k plus proches voisins.

La fonction `modification_liste_tuples_triée` ci-dessous prend en paramètres
 - une liste `liste_tuples` de tuples de deux éléments : le premier étant un nombre entier positif ou nul et le second une chaîne de caractères ; cette liste est triée par ordre croissant sur le premier élément des tuples ;
 - un tuple `tuple_à_insérer` de deux éléments : le premier étant un nombre entier positif ou nul et le second une chaîne de caractères.
Cette fonction renvoie la liste `liste_tuples` dans laquelle le tuple `tuple_à_insérer` a été inséré au bon indice de façon à conserver une liste triée par rapport au premier élément de chaque tuple.

_On rappelle que si `L` est une liste, alors l'instruction `L.insert(i, elem)` insère dans `L` l'élément `elem` a l'indice `i`._<br>
_Par exemple, si `L = [0, 3, 4]`, alors `L.insert(1, 5)` modifie la liste `L` en la liste `[0, 5, 3, 4]`._

In [ ]:
def modification_liste_tuples_triée(liste_tuples, tuple_à_insérer):
    if liste_tuples == []: # si la liste est vide
        liste_tuples.append(tuple_à_insérer) # on insère directement le tuple
    else: # sinon
        indice_insertion = len(liste_tuples) # on cherche la position à insérer à partir de la fin
        # on teste à chaque étape la valeur de la position d'insertion et
        # on compare la première valeur du tuple à insérer avec la première valeur du tuple précédent la position
        # possible d'insertion
        while indice_insertion > 0 and liste_tuples[indice_insertion - 1][0] > tuple_à_insérer[0]:
            indice_insertion = indice_insertion - 1 # on diminue l'indice de la position d'insertion
        liste_tuples.insert(indice_insertion, tuple_à_insérer)
    return liste_tuples

Les quatre cellules ci-dessous permettent de tester ce code. Elles doivent toutes renvoyer `True` lors de leur exécution.

In [ ]:
liste_initiale = []
valeur_à_insérer = (12, 'a')
liste_finale = modification_liste_tuples_triée(liste_initiale, valeur_à_insérer)
liste_finale == [(12, 'a')]

In [ ]:
liste_initiale = [(12, 'a')]
valeur_à_insérer = (18, 'b')
liste_finale = modification_liste_tuples_triée(liste_initiale, valeur_à_insérer)
liste_finale == [(12, 'a'), (18, 'b')]

In [ ]:
liste_initiale = [(12, 'a'), (18, 'b')]
valeur_à_insérer = (5, 'c')
liste_finale = modification_liste_tuples_triée(liste_initiale, valeur_à_insérer)
liste_finale == [(5, 'c'), (12, 'a'), (18, 'b')]

In [ ]:
liste_initiale = [(5, 'c'), (12, 'a'), (18, 'b')]
valeur_à_insérer = (12, 'd')
liste_finale = modification_liste_tuples_triée(liste_initiale, valeur_à_insérer)
liste_finale == [(5, 'c'), (12, 'a'), (12, 'd'), (18, 'b')]

Sur quel principe de tri classique est basée la fonction `modification_liste_tuples_triée` ?

<p style="font-style:italic; color:gray;">Ecrire la réponse ici</p>

<h2 class="alert alert-info">Exercice 4 : les k plus proches voisins</h2>

1. La première étape est d'implémenter une fonction `liste_infos_pour_décision` dont les spécifications sont les suivantes :
 - elle prend en paramètre une liste `liste_apprentissage` du type de celle de l'exercice 1 et des coordonnées GPS `coord_gps` données sous la forme d'un tuple de deux nombres flottants ;
 - elle calcule, pour chacun des éléments de la liste d'apprentissage, un tuple contenant sa distance avec le point de coordonnées GPS `coord_gps` et son département ;
 - elle renvoie une liste comprenant l'ensemble de tous ces tuples, triés par ordre croissant sur les distances.
 
Compléter cette fonction.

In [ ]:
def liste_infos_pour_décision(liste_apprentissage, coord_gps):
    liste_distances = ... # on initialise la liste à renvoyer avec la liste vide
    for elem in liste_apprentissage: # on parcourt la liste d'apprentissage par élément
        elem_gps = coordonnees_gps_formatees(elem[...]) # on formate les coordonnées GPS de elem
        elem_info = (distance(..., ...), ...) # on construit le tuple demandé
        liste_distances = ... # on insère correctement le tuple
    return liste_distances

Les trois cellules ci-dessous permettent de tester votre code. Elles doivent toutes renvoyer `True` lors de leur exécution.

In [ ]:
# La ville de Lille est présente dans la liste d'apprentissage
coord_gps_Lille = (50.6317183168, 3.04783272312)
liste_pour_décision_Lille = liste_infos_pour_décision(liste_apprentissage, coord_gps_Lille)
liste_pour_décision_Lille[:10] == [(0, 'NORD'), (3, 'NORD'), (4, 'NORD'), (4, 'NORD'), (4, 'NORD'), 
                             (5, 'NORD'), (6, 'NORD'), (6, 'NORD'), (6, 'NORD'), (6, 'NORD')]

In [ ]:
# La ville d'Aubergenville est présente dans la liste d'apprentissage
coord_gps_Aubergenville = (48.9626916321, 1.84870251915)
liste_pour_décision_Aubergenville = liste_infos_pour_décision(liste_apprentissage, coord_gps_Aubergenville)
liste_pour_décision_Aubergenville[:30] == [(0, 'YVELINES'), (1, 'YVELINES'), (2, 'YVELINES'), (3, 'YVELINES'), (4, 'YVELINES'),
                             (4, 'YVELINES'), (4, 'YVELINES'), (5, 'YVELINES'), (6, 'YVELINES'), (6, 'YVELINES'),
                             (6, 'YVELINES'), (7, 'YVELINES'), (7, 'YVELINES'), (7, 'YVELINES'), (7, 'YVELINES'),
                             (8, 'YVELINES'), (8, 'YVELINES'), (9, 'YVELINES'), (9, 'YVELINES'), (9, 'YVELINES'),
                             (9, 'YVELINES'), (9, 'VAL D OISE'), (10, 'YVELINES'), (10, 'YVELINES'), (10, 'YVELINES'),
                             (11, 'YVELINES'), (11, 'YVELINES'), (12, 'YVELINES'), (12, 'YVELINES'), (12, 'YVELINES')]

In [ ]:
# La ville de Carrières-sur-Seine n'est pas présente dans la liste d'apprentissage
coord_gps_Carrières = (48.916672, 2.18333)
liste_pour_décision_Carrières = liste_infos_pour_décision(liste_apprentissage, coord_gps_Carrières)
liste_pour_décision_Carrières[:50] == [(2, 'HAUTS DE SEINE'), (4, 'HAUTS DE SEINE'), (4, 'YVELINES'), (4, 'YVELINES'),
                             (5, 'VAL D OISE'), (6, 'YVELINES'), (7, 'HAUTS DE SEINE'), (7, 'HAUTS DE SEINE'),
                             (8, 'VAL D OISE'), (8, 'HAUTS DE SEINE'), (8, 'YVELINES'), (8, 'PARIS'), (8, 'YVELINES'),
                             (9, 'YVELINES'), (9, 'VAL D OISE'), (9, 'VAL D OISE'), (9, 'HAUTS DE SEINE'),
                             (9, 'PARIS'), (10, 'VAL D OISE'), (10, 'HAUTS DE SEINE'), (10, 'SEINE SAINT DENIS'),
                             (10, 'YVELINES'), (11, 'YVELINES'), (11, 'PARIS'), (11, 'PARIS'), (11, 'YVELINES'),
                             (11, 'YVELINES'), (12, 'YVELINES'), (12, 'VAL D OISE'), (12, 'PARIS'), (12, 'VAL D OISE'),
                             (12, 'YVELINES'), (12, 'PARIS'), (12, 'VAL D OISE'), (12, 'VAL D OISE'), (13, 'YVELINES'),
                             (13, 'HAUTS DE SEINE'), (13, 'YVELINES'), (13, 'YVELINES'), (13, 'YVELINES'),
                             (14, 'HAUTS DE SEINE'), (14, 'PARIS'), (14, 'YVELINES'), (14, 'VAL D OISE'),
                             (14, 'YVELINES'), (14, 'YVELINES'), (14, 'HAUTS DE SEINE'), (15, 'VAL D OISE'),
                             (15, 'YVELINES'), (15, 'YVELINES')]

2. Une fois que l'on a construit notre liste contenant les informations nécessaires à la décision, il faut regarder les $k$ premiers éléments de cette liste et récupérer le nombre fois où chaque département présent est obtenu. Pour cela, on construit un dictionnaire dont les clés sont les départements présents dans les $k$ premiers éléments et les valeurs correspondent au nombre de fois où ces départements apparaissent.

La fonction ci-dessous, qui est à compléter, prend en paramètre un entier naturel non nul `k` et la liste des informations nécessaires à la décision, et renvoie le dictionnaire souhaité.

In [ ]:
def dictionnaire_infos_pour_décision(k, liste_pour_décision):
    dico_département = {} # on initialise le dictionnaire à renvoyer avec le dictionnaire vide
    for i in range(...): # on parcourt les k premiers éléments de la liste
        if liste_pour_décision[i][...] in dico_département:
            dico_département[...] = dico_département[...] + ...
        else:
            ...
    return dico_département

Les trois cellules ci-dessous permettent de tester votre code. Elles doivent toutes renvoyer `True` lors de leur exécution.

In [ ]:
dictionnaire_pour_décision_Lille = dictionnaire_infos_pour_décision(7, liste_pour_décision_Lille)
dictionnaire_pour_décision_Lille == {'NORD': 7}

In [ ]:
dictionnaire_pour_décision_Aubergenville = dictionnaire_infos_pour_décision(35, liste_pour_décision_Aubergenville)
dictionnaire_pour_décision_Aubergenville == {'YVELINES': 34, 'VAL D OISE': 1}

In [ ]:
dictionnaire_pour_décision_Carrières = dictionnaire_infos_pour_décision(75, liste_pour_décision_Carrières)
dictionnaire_pour_décision_Carrières == {'HAUTS DE SEINE': 12, 'YVELINES': 27, 'VAL D OISE': 22, 'PARIS': 7,
                                         'SEINE SAINT DENIS': 5, 'VAL DE MARNE': 2}

3. Ce dictionnaire étant construit, il ne reste plus qu'à renvoyer la liste des clés correspondant à la valeur maximale des valeurs.

Compléter la fonction ci-dessous prenant en paramètre un dictionnaire comme précédemment et renvoyant la liste voulue.

In [ ]:
def liste_départements_pour_décision(dictionnaire_pour_décision):
    valeur_max = max(...) # on récupère la valeur maximale des valeurs du dictionnaire
    liste_départements = [] # on initialise la liste à renvoyer avec la liste vide
    for clé in dictionnaire_pour_décision:
        if dictionnaire_pour_décision[clé] == ...:
            liste_départements.append(...)
    return liste_départements

Les trois cellules ci-dessous permettent de tester votre code. Elles doivent toutes renvoyer `True` lors de leur exécution.

In [ ]:
liste_départements_pour_décision(dictionnaire_pour_décision_Lille) == ['NORD']

In [ ]:
liste_départements_pour_décision(dictionnaire_pour_décision_Aubergenville) == ['YVELINES']

In [ ]:
liste_départements_pour_décision(dictionnaire_pour_décision_Carrières) == ['YVELINES']

4. Nous pouvons à présent implémenter une fonction `décision_k_plus_proches_voisins` prenant en paramètre une liste `liste_apprentissage` du type de celle de l'exercice 1, des coordonnées GPS `coord_gps` données sous la forme d'un tuple de deux nombres flottants et un entier naturel non nul `k`, et renvoyant la prédiction recherchée sous la forme d'une liste de départements possibles.

In [ ]:
def décision_k_plus_proches_voisins(liste_apprentissage, coord_gps, k):
    # code à compléter

Les trois cellules ci-dessous permettent de tester votre code. Elles doivent toutes renvoyer `True` lors de leur exécution.

In [ ]:
# La ville de Lille est présente dans la liste d'apprentissage
coord_gps_Lille = (50.6317183168, 3.04783272312)
décision_k_plus_proches_voisins(liste_apprentissage, coord_gps_Lille, 7) == ['NORD']

In [ ]:
# La ville d'Aubergenville est présente dans la liste d'apprentissage
coord_gps_Aubergenville = (48.9626916321, 1.84870251915)
décision_k_plus_proches_voisins(liste_apprentissage, coord_gps_Aubergenville, 35) == ['YVELINES']

In [ ]:
# La ville de Carrières-sur-Seine n'est pas présente dans la liste d'apprentissage
coord_gps_Carrières = (48.916672, 2.18333)
décision_k_plus_proches_voisins(liste_apprentissage, coord_gps_Carrières, 75) == ['YVELINES']

5. Exécuter les cellules suivantes. Qu'en déduit-on ?

In [ ]:
coord_gps_Carrières = (48.916672, 2.18333)
décision_k_plus_proches_voisins(liste_apprentissage, coord_gps_Carrières, 5)

In [ ]:
coord_gps_Carrières = (48.916672, 2.18333)
décision_k_plus_proches_voisins(liste_apprentissage, coord_gps_Carrières, 19)

In [ ]:
coord_gps_Carrières = (48.916672, 2.18333)
décision_k_plus_proches_voisins(liste_apprentissage, coord_gps_Carrières, 31)

<p style="font-style:italic; color:gray;">Ecrire la réponse ici</p>

<h2 class="alert alert-info">Exercice 5 : applications</h2>

En utilisant la fonction précédente, prédire les départements des communes suivantes :
 - Vannes
 - Nancy
 - Lanslevillard
 - Houilles
 - Tours

On cherchera les coordonnées GPS sur le Web et on fera une vérification du département.

In [ ]:
# Vannes

In [ ]:
# Nancy

In [ ]:
# Lanslevillard

In [ ]:
# Houilles

In [ ]:
# Tours

<a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Licence Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a><br />Ce document  est mis à disposition selon les termes de la <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/">Licence Creative Commons Attribution -  Partage dans les Mêmes Conditions 4.0 International</a>.
Pour toute question : <a href="mailto:charles.poulmaire@ac-versailles.fr">charles.poulmaire@ac-versailles.fr</a> ou <a href="mailto:pascal.remy@ac-versailles.fr">pascal.remy@ac-versailles.fr</a>